- Title: QC and Join data in Arcpy
- Slug: arcpy-qc-and-join
- Category: ArcPY
- Date: 2024-11-01
- modified 2025-03-14
- Tags: Python, Arcpy, GIS
- Author: Brian Estevez
- Summary: In this post we walkthrough some QC checks, accessing attributes and geometry, and a spatial join using Arcpy


<img src= "{static}/images/ArcPY_QC_FlowChart.svg" alt ="Obsidian-generated flow chart depicting steps of an Arcpy spatial data QC and join" style= " width 400px; height: 400px;">

- Flow chart of a workflow to prepare two datasets for spatial join
- The analysis includes checks for unique ID fields and duplicate geometry 

### **What is Arcpy?:**

Arcpy is a Python library for automating GIS tasks in Arc GIS Pro
- Collection of modules, functions, and classes from the ArcGIS toolbox
- It allows you to perform geoprocessing, analysis, data management and mapping automation using python

In [ ]:
import arcpy # allows for access to ArcGIS pro geoprocessing tools and workflow automation
import os # enables interaction with local system resources (paths to folders and files)
import requests # access data from the web
import zipfile ##  process and extract downloaded zipfiles 
import pandas as pd ## use to check spreadsheet formatting


initial_dir= os.getcwd() # capture the initial directory before setting environment for arcpy
print("Initial working directory: ", initial_dir)

Initial working directory:  c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY


In [3]:
### BE SURE TO INSTALL MAGIC LIBRARY FIRST for filetype validation example below : 
# step1 activate arcgispro-py3 environment: conda activate "C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3"
# step2 install library: pip install python-magic-bin on windows, for unix system use conda install -c conda-forge python-magic

import magic ## to validate file type requested from web



### What we did last time
- Used requests libary to pull a shapefile dataset of ground water contamination from MN Geospatial Commons
- Used arcpy.conversion.FeatureClassToGeodatabase() to load a list of shapefiles into a geodatabase
- Created a function that would simplify checking the existance and characteristics of our feature classes

### **What we will do this time:**
- Revisit the custom function to check our gdb
- Create a new function using the request library
- Pull some new data into our project
- Perform several QC checks on attributes and geometry
- Perform Spatial Analysis within Arcpy

### **Create and re-use Custom Functions to Save time**
- When you create a python script, the functions within those scripts can be loaded into other scripts
    - To import your script, do as you would any library in python
        - ```python import my_script as ms```

    - Now the functions within this script are available to use 
        - ```python ms.my_function()```
- Not used here but when multiple script hold different tools you'd be better served using a package and `__init__.py` setup

In [3]:
## Let's load in our custom tool
import custom_arcpy_tools as cat


Initial working directory:  c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY


In [ ]:
# note: When you import the custom arcpy tools module it sets a variable called initial_dir to store the path where your script is located
        # this is intentionally done to support the modules functionality i.e., creating folders locally from a known reference point to store data

In [4]:
cat.listFC_dataset_Attributes()

## We purposefully caused an error!
# Why? To demonstrate that the error is due to workspace not being set in this new notebook or script

ValueError: No workspace is set! Please specify a workspace

In [ ]:
# See the ValueError? 
### Don't remember the arguments for this function? 
### Use this:
#  help(function_name)

In [5]:
help(cat.listFC_dataset_Attributes) ## use help(function_name) to get the documentation for the function 

Help on function listFC_dataset_Attributes in module custom_arcpy_tools:

listFC_dataset_Attributes(workspace=None)
    List feature classes in the workspace and checks their key attributes, such as SpatialReference , 
    geometry type, and coordinate systems
    Parameters:
    -----------
    workspace: str, optional
        The file path to the workspace (e.g., geodatabase path) containing the feature classes
        If none, the function will use the current ArcPy workspace
        If no workspace is set will raise an error telling you to set a a workspace
    
    Returns:
    --------
    None , if no workspace exists
    Otherwise,    Prints the details of all feature classes in the wksp , including
                - Feature Class Name
                - spatial reference name
                - spatial reference type
                - Geometry
                - Well-known ID of the spatial reference
                Also, warns if different spatial references or coordinate system

In [5]:
# Let's change the directory to where we have a geodatabase with some feature classes to work with
# Note: % is a magic command to access local system file and folders
%cd ./02 Result 

c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY\02 Result


In [6]:
# list the files
%ls 

 Volume in drive C is OS
 Volume Serial Number is DA19-2472

 Directory of c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY\02 Result

03/21/2025  10:29 AM    <DIR>          .
03/28/2025  03:15 PM    <DIR>          ..
03/29/2025  05:47 PM    <DIR>          MN_WaterContamination.gdb
               0 File(s)              0 bytes
               3 Dir(s)  437,112,291,328 bytes free


In [ ]:
## should see a file ending with .gdb. Copy the folder path and add its name below

In [7]:

## Use the gdb name in the folder, and set the path to the gdb

###======================    Set up file paths


gdb_path = r"c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY\02 Result\MN_WaterContamination.gdb"

##===================== Define the path to script and project folder

# set the folder path as the working environment, and store it for later use
try:
    arcpy.env.workspace = gdb_path
    wksp = arcpy.env.workspace
except Exception as e:
    print(f"Error setting up the path, {e}, Exception type: {type(e).__name__}") # python's built-in error messaging, {e} prints description and type(e).__name__ category

print("Working environment is here",wksp)

## Modify the env attribute to allow overwriting output files
arcpy.env.overwriteOutput= True

Working environment is here c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY\02 Result\MN_WaterContamination.gdb


In [ ]:
arcpy.Exists(wksp) ## similar to using print statement following if not os.path.exists(gdb_path)

True

In [9]:
# Now. Retry the function that failed earlier

cat.listFC_dataset_Attributes()

NameError: name 'cat' is not defined

In [10]:
# Confirm we have those feature classes
arcpy.ListFeatureClasses()

['atlas_contamination_areas',
 'atlas_contamination_area_lines',
 'atlas_contamination_sites',
 'atlas_contamination_sites_noca',
 'atlas_contamination_source_areas',
 'atlas_contamination_wells_result_sum',
 'MN_PFAS_Levels',
 'MN_PFAS_Levels_proj',
 'MN_PFAS_Levels_by_City',
 'city_township_unorg',
 'overlap_check1',
 'MN_WaterContamination_SpatialJoin',
 'MN_latest_WaterSamples_Joined_lyr']

### ***Data Sources*** :
-  Geography: MN

|Input Name |file types|Coordinate System| Source|
|-----------|-----------|------------------------------------|--------------------------------|
|MN PFAS levels| 1 excel file |GCS: NAD 83|[Report](https://www.pca.state.mn.us/sites/default/files/w-sw4-37.pdf)
|MN Groundwater contamination atlas | 6 shape files |PCS: NAD 83 UTM Zone 15N|[MetaData](https://resources.gisdata.mn.gov/pub/gdrs/data/pub/us_mn_state_pca/env_mn_gw_contamination_atlas/metadata/atlas_wells_result_summary.html)|
|MN Cities and townships| 1 shapefile|PCS: NAD 83 UTM Zone 15N|[MetaData](https://webgis.dot.state.mn.us/65agsf1/rest/services/sdw_govnt/CITY_TOWNSHIP_UNORG_TERR/FeatureServer)
- We have already pulled and done some initial processing of all these datasets in the previous three walkthroughs
- Now will validate uniqueness of the data and join the datasets

In [11]:
# quick reminder of where our current workspace is set
print("Data Folder is here" , wksp)
# let's call it gdb for clarity
gdb=wksp

Data Folder is here c:\Projects\my_git_pages_website\Py-and-Sky-Labs\content\ArcPY\02 Result\MN_WaterContamination.gdb


### **Preparing for Spatial Join by understanding the Grain of the Data**

- When joining any datasets, spatial or otherwise, it is a good practice to:

    - ***Validate the record counts of the initial and joined datasets***
    - ***Understand the grain of the datasets***
        - In our case we have two datasets at different levels of detail or resolution:
            -  one dataset is organized by City,Townships, and Unorganized territories (CTU) that are each assigned a unique ID called  Geographic Names Information System Feature ID (GNIS_Feature_ID)
            - CTU layer **grain = an individual city/township/territory polygon** uniquely identified by GNIS_Feature_ID
                    - A unique ID from the GNIS federal database developed by USGS
                    - GNIS database contains official names and GNIS_Feature_ID for geographic features (cities, townships, lakes, etc)
            - The other dataset is of organized by unique well location IDs called sys_loc_code
            - Well data **grain = an individual well location point** , uniquely identified by sys_loc_co
        - Why does this matter?: 
            - A unique standard ID should identify the grain of a dataset--> 'thing' each unique row or geometry represents
            - Use that ID for analysis of discrete entities (e.g., cities) 
                - e.g., Each GNIS_Feature_ID corresponds to a single unique location (City,Township or Unorganized Territory)
            - Spatial join can move the data from one grain to another
                - individual well --> well + CTU polygon
        - What it helps with
            - understanding how datasets relates
            - avoiding incorrect joins
            - preventing data summary/aggregation errors


####  ***Common Unique ID fields and What they represent***:
|ID Field| Description|Grain/Entity|
|--------|------------|------------|    
|GNIS_Feature_ID| USGS ID for natural cultural features| City, township, water body|
|GEOID | US Census Bureau geocode | Tract, block group , place|
|FIPS| Federal Information Processing Standard| State, county , subdivision|
|UUID| Universally Unique ID (internal system generated) | Custom entity define by creator| 
|PLACE_ID| Google Unique ID for places | Point of Interest|
|sys_loc_co| Unique well / location ID |Groundwater sampling|

- These IDs define the entities each dataset represents
- When joining, aggregating or visualizing need to correctly use these IDs to avoid mixing grain or double counting

#### MN City,Township, Unorganized Territory (CTU) Dataset
- Represents boundaries of cities, townships and unorganized territories in Minnesota
- key fields
    - GNIS_Feature_ID: 
        - official ID for geographic features (e.g., cities) that relates them to their Name and location in the GNIS Database
    - County_GNIS_Feature_ID: 
        - official county ID that relates counties to their Name and Location in the GNIS Database
    - FEATURE_NAME: 
        - Name of the city, township or unorganized territory
- Source : MN Geospatial Commons: 
    - url: https://gisdata.mn.gov/dataset/11860ca6-9ee4-4ff2-b30b-da9e2f13262a

- #### **What is the grain of the CTU Polgon layer? How does it Relate to the grain of the wells Layer?**

In [12]:
## Count initial records
# first check the join fc: County/Township/Unorganized territory (CTU)

fc= "city_township_unorg"

print(f"Total records in {fc} feature class is:  {int(arcpy.management.GetCount(fc)[0])}")

Total records in city_township_unorg feature class is:  2744


In [ ]:
### Based on the metadata, this dataset is supposed to represent city/township/terr (CTU) boundaries across the state of MN
#  Grain of the data is at the individual CTU level, each row is likely to be represented by a unique polygon
    #  Does each row have a corresponding unique ID? 
    #  Are there overlapping CTUs?

In [13]:
## check the target input fc

for fc in arcpy.ListFeatureClasses("*wells_result_sum*"):
    print(f"Feature class: {fc}\n Total Record Count: {int(arcpy.management.GetCount(fc)[0])}")

Feature class: atlas_contamination_wells_result_sum
 Total Record Count: 17443


In [ ]:
### Based on the metadata the dataset represents well locations
#  Grain is at the level of individual wells, each row is likely to be represented by a unique point
    # Does each row have a corresponding unique ID?
    # Are the same wells sampled multiple times on the same date?

In [14]:
## List and summarize all fields

print("Summary of Fields")

for field in arcpy.ListFields("city_township_unorg"):
                    # For better display left align, pad to same length. e.g, field.name:20, means left align text and pad to 20 characters
    print(f" - {field.name:15} | {field.type:10} | {field.length}") 

Summary of Fields
 - OBJECTID        | OID        | 4
 - Shape           | Geometry   | 0
 - GNIS_FEATU      | Integer    | 4
 - FEATURE_NA      | String     | 254
 - CTU_CLASS       | String     | 25
 - COUNTY_GNI      | Integer    | 4
 - COUNTY_COD      | String     | 2
 - COUNTY_NAM      | String     | 100
 - POPULATION      | Integer    | 4
 - SHAPE_Leng      | Double     | 8
 - Shape_Length    | Double     | 8
 - Shape_Area      | Double     | 8
 - Combined_GNIS_IDs | String     | 50


In [15]:
### lets wrap this in a function and add total record counts
# A function to get the fields and record counts

def showFieldinfo(fc):
    record_count = arcpy.management.GetCount(fc)[0]
    
    print("Summary of Fields")

    for field in arcpy.ListFields(fc):
        print( f" {field.name:15} | {field.type:15} | {field.length}")
        
    print(f"\nRecord Count:{record_count} in Feature Class {fc}")
    return

In [16]:
fc = "city_township_unorg"
showFieldinfo(fc)

Summary of Fields
 OBJECTID        | OID             | 4
 Shape           | Geometry        | 0
 GNIS_FEATU      | Integer         | 4
 FEATURE_NA      | String          | 254
 CTU_CLASS       | String          | 25
 COUNTY_GNI      | Integer         | 4
 COUNTY_COD      | String          | 2
 COUNTY_NAM      | String          | 100
 POPULATION      | Integer         | 4
 SHAPE_Leng      | Double          | 8
 Shape_Length    | Double          | 8
 Shape_Area      | Double          | 8
 Combined_GNIS_IDs | String          | 50

Record Count:2744 in Feature Class city_township_unorg


In [ ]:
## We now have a way to summarize all fields in a more visual way

### **QC Overlapping Polygons: Do CTU polygons overlap?**
- This is to ensure that we do have a unique polygon for each CTU
- If we do, bucketing well contamination sites by unique CTU will make sense
- otherwise, same wells would be grouped into different CTU polygons causing misinterpretation of wells data per CTU

#### Add a positive control for overlap
- we will grab a random polygon from CTU layer using the SearchCursor and the SHAPE@WKT field(i.e., string of polygon shape vertices coordinates)
- Then insert the polygon into the same CTU layer to create a duplicate
- This way we know we have an overlap and that our method can detect overlap


In [19]:
# We can even access the geometry of the layer
fc = "city_township_unorg"

counter= 0
with arcpy.da.SearchCursor(fc, ["SHAPE@WKT"]) as cursor:
    for row in cursor:
        while counter <5:
            shape= row[0]
            counter +=1
            print(shape)

MULTIPOLYGON (((230665.87399999984 5383161.6950000003, 230639.37100000028 5383162.7919999994, 230554.57699999958 5383166.5490000006, 230412.21700000018 5383172.8640000001, 230267.49100000039 5383179.2809999995, 230107.56300000008 5383186.3880000003, 229883.49500000011 5383196.3269999996, 229855.51300000027 5383197.6449999996, 229756.63499999978 5383202.0010000002, 229443.33800000045 5383215.8969999999, 229136.76300000027 5383229.5449999999, 229047.27300000004 5383233.6539999992, 227613.41100000031 5383306.2300000004, 227435.81099999975 5383315.2200000007, 225815.58100000024 5383386.5600000005, 224217.28699999955 5383459.8330000006, 224207.0700000003 5383460.3010000009, 223712.59499999974 5383488.3249999993, 223435.43599999975 5383503.9979999997, 223216.95799999963 5383516.3440000005, 223077.14699999988 5383522.7039999999, 222604.60599999968 5383544.1999999993, 222530.85400000028 5383547.6620000005, 221193.40799999982 5383608.6380000003, 220996.10800000001 5383617.6520000007, 220984.069

### **More on Well-Known Text(WKT) format data**
`SHAPE@WKT`
- WKT is a text-based format standardized by the international Open GeoSpatial Consortium of experts on geospatial data standards
    - general format Point (x,y)
        - If CRS is geographic (x,y) --> longitude , latitude
        - If Projected         (x,y)  --> easting, northing
- Used to represent the vertices for vector geometry or describe coordinate systems
- Two types of WKT:
    1. WKT for ***Geometry*** (most commonly used):
        - Point (30 10) 
        - LINESTRING (30 10, 10 30, 40 40)
        - POLYGON (30 10, 40 40, 20 40, 10 20, 30 10)
        - MULTIPOLYGON ((...)) .. same as polygon but nested as multiple polygons
    2. WKT for ***Coordinate Systems***
        - GEOGCS["WGS 84", DATUM["WGS_1984",...]]
- WKT is used in:
    - PostGIS (PostgreSQL extension) to insert, export and transform geometries
        - PostGIS also supports well-known binary format (WKB) but WKT is more human-readable
    - APIs / Data Interchange to send, receive data from spatial services, GeoJson to WKT conversion, import export format in tools likeArcGIS, QGIS
    - Although WKT is vector data, can be used to represent raster extent, projection system (e.g., might be present in raster metadata for spatial reference)

In [ ]:
## Copy a polygon WKT from the above output and paste it here
## Your copied polygon may be different from the one shown here, works the same regardless
### MULTIPOLYGON (((353468.7034 4904892.6543000005, 353482.93570000026 4905697.5601000004, 351878.13520000037 4905722.3309000004, 351869.37359999958 4904918.1735999994, 351853.66249999963 4904119.7785999998, 353448.25569999963 4904099.7821999993, 353468.7034 4904892.6543000005)))

In [ ]:
### Note that the default ArcPY behavior for shapefiles labels WKT polygons as multipolygons
### The main benefit of WKT is being easy to visually see and understand, much easier to work with and move between systems

In [ ]:
### Say you want the Geometry Type of the features in WKT format
from collections import Counter

fc = "city_township_unorg"
geom_list = set()

with arcpy.da.SearchCursor(fc, ["SHAPE@WKT"]) as cursor:
    for row in cursor:
        geom = row[0]
        if geom:
            geom_name = str(geom).split()[0] # isolates the name from the WKT
            geom_list.add(geom_name)

print("List of WKT Geometries\n", geom_list)

count_geom_types  =  Counter(geom_list) # counter creates a dictionary with keys and counts of keys as the values

for key,count in count_geom_types.items(): # .items() dictionary method
    if int(count) >0:
        print(key, count) #  because we made this based off unique set list of names, we should see different names if they exist and their count


List of WKT Geometries
 {'MULTIPOLYGON'}
MULTIPOLYGON 1


### **SHAPE@ ArcGIS Native Geometry Object**
#### `SHAPE@`
- Not a string but a geometry object
- Has special methods and properties 
- Can access:
    - .area, .length, .centroid, .extent, .partCount

- Can use to construct buffers, intersection

In [ ]:
from collections import Counter


geom_list= set()
with arcpy.da.SearchCursor(fc, ["SHAPE@"]) as cursor:
    for row in cursor:
        geom =   row[0]
        if geom:
            geom_list.add(geom.type) # Can access other properties of geometry object : .area, .centroid, .extent, .partCount

count_geom_types = Counter(geom_list)

for key, count in count_geom_types.items():
    if count>0:
        print(key, count)


polygon 1


In [ ]:
### In both examples, there is only one geometry type for the CTU dataset, as expected 

### **Manually add Geometry features to a layer with `arcpy.managment.Append()` and `arcpy.FROMWKT()`**
#### `arcpy.management.Append()`
- Adds new features to an **existing dataset**
- Can add point, line or polygon feature classes to an **existing dataset of the same type** ( e.g., cannot add line fc to a point fc)
- Can also add several tables to an existing table or several rasters to an existing raster
- arcpy.management.Append(inputs, target, {schema_type}, {field_mapping}, {subtype}, {expression}, {match_fields}, {update_geometry}, {enforce_domains})
- key fields: 
    - input: data to be appended to the target dataset (Note you can combine Tables and Feature Classes e.g., Fc+Table--> NewTable with Fc attributes)
    - target: existing dataset where the input will be added
    - schema_type: whether fields must match. TEST, causes error if fields do not match. NO_TEST allows fields to be null,TEST_AND_SKIP, will leave out the non-matching fields

#### ` arcpy.FromWKT(wkt_string,{spatial_reference})`
- Creates a geometry object from well-known text string
- This geometry object is the same one as the native SHAPE@

#### Use-case
- Updating a polygon layer with a new feature (e.g., add a new city, township, territory to an existing CTU polygon layer)
- Consolidate multiple datasets into one
- Update a table with new , more recent data (e.g., update site sampling data for an existing feature)

In [ ]:
# Insert a single duplicate polygon to CTU polygon layer
    # Why?: a positive control to test for overlapping polygons, we already know it will overlap but we really want to know that our detection methods work too

wkt_geom= """MULTIPOLYGON (((353468.7034 4904892.6543000005, 353482.93570000026 4905697.5601000004, 351878.13520000037 4905722.3309000004, 351869.37359999958 4904918.1735999994, 351853.66249999963 4904119.7785999998, 353448.25569999963 4904099.7821999993, 353468.7034 4904892.6543000005)))"""
fc = "city_township_unorg"

try:
    ### TOOL: arcpy.management.Append(inputs, target, {schema_type}, {field_mapping}, {subtype}, {expression}, {match_fields}, {update_geometry}, {enforce_domains})
    arcpy.management.Append(inputs=[arcpy.FromWKT(wkt_geom,arcpy.Describe(fc).spatialReference)], target=fc, schema_type="NO_TEST")
                                ## Schema_type is whether you want to enforce field matching for the inserted geometry. If TEST, all fields must match exactly, including types and order
                                ## we just have the WKT so we do not want to enforce field matching(NO_TEST)

except arcpy.ExecuteError:
    print("Arcpy Error:", arcpy.GetMessages(2))

except Exception as e:
    print("Error occurred: ", e, type(e).__name__)

In [ ]:
# recall we can access fc attributes using describe object
# in the previous example we accessed the CTU attributes to set the spatial reference of the new polygon geometry object
sr= arcpy.Describe(fc).spatialReference

print(sr.name, sr.factoryCode)



NAD_1983_UTM_Zone_15N 26915


### **Use PairwiseIntersect to detect overlapping Features**
#### `arcpy.analysis.PairwiseIntersect()`
- arcpy.analysis.PairwiseIntersect(in_features, out_feature_class, {join_attributes}, {cluster_tolerance}, {output_type})
- Computes all possible pairings
    - self:self  (FeatureA: FeatureA)
    - self:other (FeatureA: FeatureB)
    - other:self (FeatureB: FeatureA)

- For this QC exercise, only self:other pairing is meaningful:
    - Filter-out self:self pairs (Features pairing with themselves)
    - Select only self:other (Features pairing with non-self features)
    - Keep only one direction of pairing (e.g., FID > FID_1 or FID < FID_1 --both mean that the match has different object IDs and overlap)

In [26]:
# Check for overlap within a feature class of CTU polygons

fc = "city_township_unorg"
in_fc = fc 
field_name= f"FID_{fc}"
where_c = f"FID_{fc} > FID_{fc}_1"
out_fc = "overlap_check1"
overlapping_ids_list= set()

try:
### TOOL: arcpy.analysis.PairwiseIntersect(in_features, out_feature_class, {join_attributes}, {cluster_tolerance}, {output_type})
    arcpy.analysis.PairwiseIntersect([in_fc, in_fc], out_fc, join_attributes="ONLY_FID") # We only want FID to track which of the original ObjectIDs overlap
    print("Successfully completed PairwiseIntersect Analysis")

    with arcpy.da.SearchCursor(out_fc,[field_name],where_clause=where_c) as cursor:
        for row in cursor:
            dup_id = row[0]
            overlapping_ids_list.add(dup_id)

### =========================   Filter for actual overlaps (not self-intersections)
### Step1: Create a temporary layer from the pairwise overlap feature class
### Step2: Select those paired features whose FID values do not match, overlapping features
### Step3: Count the number of those overlapping features

    ### TOOL: arcpy.management.MakeFeatureLayer(in_features, out_layer, {where_clause}, {workspace}, {field_info})
    arcpy.management.MakeFeatureLayer(out_fc, "overlap_lyr")

### TOOL: arcpy.management.SelectLayerByAttribute(in_layer_or_view, {selection_type}, {where_clause}, {invert_where_clause})
    arcpy.management.SelectLayerByAttribute("overlap_lyr", "NEW_SELECTION",
                                        where_c) # keep only one intersection of the pair (A->B) to clearly mark one overlap 
    print("Successfully filtered PairwiseIntersect features")
    
    count = int(arcpy.management.GetCount("overlap_lyr")[0])
    print(f" PairwiseIntersect detected {count} overlaps for the following objects Ids: \n{overlapping_ids_list}")

except Exception as e:
    print("Error Occurred", e, type(e).__name__)
except arcpy.ExecuteError:
    print("ArcPy Error", arcpy.GetMessages(2))


### We identified overlapping polygons and printed object ID  (here 2745 matched to 18)
### This is the object ID from the original CTU polygon table 
### The larger ID value is selected because features are labeled in sequence in a fc, making it likely the more recent one was unintentionally added

Successfully completed PairwiseIntersect Analysis
Successfully filtered PairwiseIntersect features
 PairwiseIntersect detected 1 overlaps for the following objects Ids: 
{2745}


In [ ]:
### We identified overlapping polygons and printed object ID 
### This is the object ID from the original CTU polygon table 
### The larger ID value is selected because features are labeled in sequence in a fc, making it likely the more recent one was unintentionally added

In [31]:
def check_Fc_NonselfOverlap(input_fc, output_fc_name):
    """ Detects overlapping features within a feature class using PairwiseIntersect.
        Returns duplicate pair object ID and count
    Args:
        input_fc (str): Full path or name of the feature class
        output_fc_name: output feature class name for PairwiseIntersect results
    Returns:
        count (int) : number of overlapping features 
        overlap_ids (list):  list of Object IDs of overlapping features 
        output_fc (str) : Name of the output overlap feature class 
    
    """
    # Check for overlap within a feature class of CTU polygons

    base_name = os.path.basename(input_fc)
    safe_name= arcpy.ValidateTableName(base_name) 
    field_name= f"FID_{safe_name}"
    field_name_1= f"FID_{safe_name}_1"
    where_c = f"{field_name} > {field_name_1}"
    out_fc = output_fc_name
    overlapping_ids_list= set()

    try:
    ### TOOL: arcpy.analysis.PairwiseIntersect(in_features, out_feature_class, {join_attributes}, {cluster_tolerance}, {output_type})
        arcpy.analysis.PairwiseIntersect([input_fc, input_fc], out_fc, join_attributes="ONLY_FID") # We only want FID to track which of the original ObjectIDs overlap
        print("Successfully completed PairwiseIntersect Analysis")

        with arcpy.da.SearchCursor(out_fc,[field_name],where_clause=where_c) as cursor:
            for row in cursor:
                dup_id = row[0]
                overlapping_ids_list.add(dup_id)

    ### =========================   Filter for actual overlaps (not self-intersections)
    ### Step1: Create a temporary layer from the pairwise overlap feature class
    ### Step2: Select those paired features whose FID values do not match, overlapping features
    ### Step3: Count the number of those overlapping features

        ### TOOL: arcpy.management.MakeFeatureLayer(in_features, out_layer, {where_clause}, {workspace}, {field_info})
        arcpy.management.MakeFeatureLayer(out_fc, "overlap_lyr")

    ### TOOL: arcpy.management.SelectLayerByAttribute(in_layer_or_view, {selection_type}, {where_clause}, {invert_where_clause})
        arcpy.management.SelectLayerByAttribute("overlap_lyr", "NEW_SELECTION",
                                            where_c) # keep only one intersection of the pair (A->B) to clearly mark one overlap 
        print("Successfully filtered PairwiseIntersect features")
        
        count = int(arcpy.management.GetCount("overlap_lyr")[0])
        print(f" PairwiseIntersect detected {count} overlaps for the following objects Ids: \n{list(overlapping_ids_list)}")

        return count, list(overlapping_ids_list), out_fc

    except Exception as e:
        print("Error Occurred", e, type(e).__name__)
    except arcpy.ExecuteError:
        print("ArcPy Error", arcpy.GetMessages(2))


### We wrapped the check for overlapping polygons in a function for repeatability
    

In [32]:
fc = "city_township_unorg"
input_fc = fc 
output_fc_name = "overlap_check2"

check_Fc_NonselfOverlap(input_fc, output_fc_name)

Successfully completed PairwiseIntersect Analysis
Successfully filtered PairwiseIntersect features
 PairwiseIntersect detected 1 overlaps for the following objects Ids: 
[2745]


(1, [2745], 'overlap_check2')

In [ ]:
### It's clear that CTU layer does not have overlapping polygons (beyond what we added as a control)
### This assures us that bucketing wells by CTU polygons is valid, will not have different wells assigned to multiple CTUs

In [ ]:
## Updated function with doc-strings to show unique values and geometries by field


def showFieldinfo(fc):
    """
    Outputs detailed field information for a given ArcGIS Feature Class.

    This function prints a summary of all fields in the provided Feature Class, including:
    - Field names
    - Field types (e.g., Text, Integer, Double)
    - Field length (applicable for Text fields)
    - Count of unique values for each field
    
    Additionally, it prints counts of total records and total unique Geometries (by Well-Known Text String) for the Feature Class

    Args:
    fc (str): 
        The path to the Feature Class to be analyzed. Must be accessible within the current ArcPy workspace.
    
    Returns:
    None
        This function prints the summary information directly to the console. 
        It does not return any values.

    Example:
    --------
    >>> showFieldinfo("C:/Path/To/YourGDB.gdb/YourFeatureClass")
    Summary of Fields in Feature Class: YourFeatureClass
    Name                 Type         Length   Unique_Values
    OBJECTID             OID          -       100          
    Name                 String       50      24           
    Total Record Count:  100
    Total Geometry Count:100
    """
    record_count = arcpy.management.GetCount(fc)[0]
    shape_type = arcpy.Describe(fc).shapeType
      
    print(f"*Summary*\n Fields in the {shape_type} Feature Class: {fc}\n")
    print(f"Name                 Type         Length   Unique_Values")


    geometry_set = set()
    with arcpy.da.SearchCursor(fc,["SHAPE@WKT"]) as cursor:
        for row in cursor:
            geometry_set.add(row[0])
    for field in arcpy.ListFields(fc):
        unique_ids = set()
        field_name= str(field.name)
        with arcpy.da.SearchCursor(fc,[field_name]) as cursor:
            for row in cursor:
                if row[0] is not None:
                    unique_ids.add(row[0])
        print( f" {field.name:20} | {field.type:10} | {str(field.length):5} | {str(len(unique_ids)):10}")
        
    print(f"\nTotal Record Count: {record_count}, Total Geometry Count(by WKT): {len(geometry_set)}")

    if record_count != len(geometry_set):
        print(f"Warning {(int(record_count)-len(geometry_set))} Potential Duplicate Features Detected in the Feature Class")
    return


In [90]:
# You may have noticed a new function called arcpy.da.SearchCursor()
# Enables us to access and read the attribute data for the feature classes
# More on that later

In [75]:
fc ="city_township_unorg"
showFieldinfo(fc)

*Summary*
 Fields in the Polygon Feature Class: city_township_unorg

Name                 Type         Length   Unique_Values
 OBJECTID             | OID        | 4     | 2744      
 Shape                | Geometry   | 0     | 2743      
 GNIS_FEATU           | Integer    | 4     | 2693      
 FEATURE_NA           | String     | 254   | 2249      
 CTU_CLASS            | String     | 25    | 3         
 COUNTY_GNI           | Integer    | 4     | 87        
 COUNTY_COD           | String     | 2     | 87        
 COUNTY_NAM           | String     | 100   | 87        
 POPULATION           | Integer    | 4     | 1312      
 SHAPE_Leng           | Double     | 8     | 2743      
 Shape_Length         | Double     | 8     | 2743      
 Shape_Area           | Double     | 8     | 2743      
 Combined_GNIS_IDs    | String     | 50    | 2743      

Total Record Count: 2744, Total Geometry Count(by WKT): 2743
Warning 1 Potential Duplicate Features Detected in the Feature Class


In [ ]:
## CTU Layer QC Results: Unique non-overlapping geometries but no corresponding unique attribute field

# We found that the CTU layer does represent unique spatially distinct locations
    # But it is NOT unique by the GNIS_Feature_ID field
    # CTU total record count is greater than GNIS_FEATU count (location ID field)

# Number of unique geometries or shapes in the layer (polygons in this case) matches total record count --except the one we added manually!
    # Each row represents a unique CTU location, as evident by the number of unique geometries and lack of overlap between them
    # but we do not have a corresponding unique attribute field marking each of those unique polygons


### **Reading Attribute data with SearchCursor**
#### `arcpy.da.SearchCursor()`
- Use to read attribute data (and optionally geometry) from a feature class or table
    - da, stands for 'data access', a newer module that is an update from the older arcpy.SearchCursor
-  **SearchCursor (in_table, field_names, {where_clause}, {spatial_reference}, {explode_to_points}, {sql_clause}, {datum_transformation}, {spatial_filter}, {spatial_relationship}, {search_order})**
    - Arguments:
        - in_table: path to input feature class (or name if workspace is set), shapefile or table
        - field_names: List or tuple of field names to search. Select all fields using * versus passing a field or list of field names
        - where_clause- SQL expression that selects a subset of records to view
        - spatial_reference- use to project/transform spatial reference of the feature class. Requires a spatial reference object
        - explode_to_points: break a multipoint fc into its points and return each one as a separate record
        - sql_clause: SQL prefix and postfix clauses to modify the records returned. 
            - SQL Prefixes: DISTINCT, TOP (only supported in SQL Server db)
            - SQL Postfixes: ORDER_BY, GROUP_BY
            - Examples: sql_clause= ("prefix","postfix")
                -  sql_clause= ("TOP 5", "ORDER BY ELEVATION DESC") ;   sql_clause=("DISTINCT STREET_NAME", None)
        - datum_transformation: if changing the projection of a fc and datum are not the same, need to specify the datum
        - spatial_filter - use a geometry object to spatially filter features. Also, need to specify spatial_relationship value
        - spatial relationship:The relationship to apply to the input and geometry used in spatial_filter
            - only applicable when using spatial_filter argument
            - INTERSECTS, OVERLAPS, TOUCHES, WITHIN, CONTAINS, CROSSES, etc. Default value is INTERSECTS
        - search_order- order in which spatial search are applied. only applies when using spatial_filter. Can only use in enterprise gdb
            - ATTRIBUTEFIRST (default value)
            - SPATIALFIRST

In [ ]:
## One more tool -- Counter
# a built-in tool to count duplicates more directly

from collections import Counter

ids= ["A11", "B22", "B22", "D12","E77","E77","E77"]

counted= Counter(ids)
print(counted)

Counter({'E77': 3, 'B22': 2, 'A11': 1, 'D12': 1})


In [ ]:
# counter is a special method for counting items
# used to count things that are in lists
    # returns a counter object that is dictionary, keys= unique values from the list, and values=how many times the key appears

In [73]:
## Check for repeated ID
from collections import Counter

fc = "city_township_unorg"


# identify the duplicates
ids = []
with arcpy.da.SearchCursor(fc, ["GNIS_FEATU"]) as cursor:
    for row in cursor:
        ids.append(row[0])

# extract the duplicate IDs into a list
duplicates = [item for item, count in Counter(ids).items() if count>1]
print(f"Duplicate ID Count: {len(duplicates)}")

# re-scan the records for the duplicate data
with arcpy.da.SearchCursor(fc,  ["GNIS_FEATU", "FEATURE_NA", "COUNTY_GNI"],sql_clause=[None,"ORDER BY GNIS_FEATU"]) as cursor:
    for row in cursor:
        if row[0] in duplicates:
            print(f"GNIS_FEATURE_ID: {row[0]} | FEATURE NAME {row[1]} | COUNTY ID {row[2]}" )

Duplicate ID Count: 48
GNIS_FEATURE_ID: 664666 | FEATURE NAME Lake City | COUNTY ID 659470
GNIS_FEATURE_ID: 664666 | FEATURE NAME Lake City | COUNTY ID 659523
GNIS_FEATURE_ID: 2393435 | FEATURE NAME Brooten | COUNTY ID 659506
GNIS_FEATURE_ID: 2393435 | FEATURE NAME Brooten | COUNTY ID 659517
GNIS_FEATURE_ID: 2393488 | FEATURE NAME Byron | COUNTY ID 659500
GNIS_FEATURE_ID: 2393488 | FEATURE NAME Byron | COUNTY ID 659465
GNIS_FEATURE_ID: 2393554 | FEATURE NAME Clearwater | COUNTY ID 659530
GNIS_FEATURE_ID: 2393554 | FEATURE NAME Clearwater | COUNTY ID 659517
GNIS_FEATURE_ID: 2393615 | FEATURE NAME Comfrey | COUNTY ID 659462
GNIS_FEATURE_ID: 2393615 | FEATURE NAME Comfrey | COUNTY ID 659453
GNIS_FEATURE_ID: 2393799 | FEATURE NAME Chanhassen | COUNTY ID 659472
GNIS_FEATURE_ID: 2393799 | FEATURE NAME Chanhassen | COUNTY ID 659455
GNIS_FEATURE_ID: 2393810 | FEATURE NAME Chatfield | COUNTY ID 659500
GNIS_FEATURE_ID: 2393810 | FEATURE NAME Chatfield | COUNTY ID 659468
GNIS_FEATURE_ID: 2394115 

In [ ]:
### Filter the table for a specific list of ids
#  print all field values for that list

from collections import Counter

fc = "city_township_unorg"

ids= [] # list ids

with arcpy.da.SearchCursor(fc, ["GNIS_FEATU"]) as cursor:
    for row in cursor:
        row_id= row[0]
        ids.append(row_id)

dup_ids =[item for item, count in Counter(ids).items() if count >1]
dup_counter = 0
with arcpy.da.SearchCursor(fc, ["*"], sql_clause =[None, "ORDER BY GNIS_FEATU"]) as cursor:
    print([field.name for field in arcpy.ListFields(fc) if field_name])
    for row in cursor:

        #  access row values that are in the duplicate id list  
        if row[2] in dup_ids: # notice how we use the row value row[2] to access the target field without having to specify its field_name directly...
            
            dup_counter += 1 
            print(row)
            
print(f"{dup_counter} total duplicate records ")

['OBJECTID', 'Shape', 'GNIS_FEATU', 'FEATURE_NA', 'CTU_CLASS', 'COUNTY_GNI', 'COUNTY_COD', 'COUNTY_NAM', 'POPULATION', 'SHAPE_Leng', 'Shape_Length', 'Shape_Area', 'Combined_GNIS_IDs']
(784, (556547.7262840756, 4923746.708489112), 664666, 'Lake City', 'CITY', 659470, '25', 'Goodhue', 904, 13416.1568982, 13416.15689824754, 2136011.176120701, '664666_659470')
(1028, (557498.3614429827, 4921089.74804861), 664666, 'Lake City', 'CITY', 659523, '79', 'Wabasha', 4480, 19234.1815176, 19234.181517616806, 9501955.626011716, '664666_659523')
(1069, (333399.3737075187, 5041338.6560099255), 2393435, 'Brooten', 'CITY', 659506, '61', 'Pope', 0, 1453.87255292, 1453.8725529151695, 54912.58523351556, '2393435_659506')
(1390, (334697.4697190577, 5040695.189919232), 2393435, 'Brooten', 'CITY', 659517, '73', 'Stearns', 646, 12517.0735421, 12517.073542148788, 4049205.4880357445, '2393435_659517')
(375, (528782.3914170366, 4876199.8177096015), 2393488, 'Byron', 'CITY', 659500, '55', 'Olmsted', 6883, 30858.991

In [ ]:
## We now validated that this GNIS_FEATURE_ID field is not uniquely identifying each location in the layer 
# (ie. some places have the same name but have/are in different counties--> aligns with why they are represented as different polygons)
    ## GNIS_FEATURE_ID: 664666 | FEATURE NAME Lake City | COUNTY ID 659470
    ## GNIS_FEATURE_ID: 664666 | FEATURE NAME Lake City | COUNTY ID 659523

## Based on this data we need to Combine GNIS_FEATURE_ID  with the County ID 
#  Can more easily work with the dataset at the level of unique locations

#### Minnesota Groundwater Contamination Atlas - Wells Contamination Results Summary Dataset
- Shows where testing results found areas of groundwater contamination
- key fields
    - Facility_ID: EQuIS facility unique row ID 
    - Sys_loc_code: Well unique identifier in EQuIS
- Source : MNPCA 
    - url: https://resources.gisdata.mn.gov/pub/gdrs/data/pub/us_mn_state_pca/env_mn_gw_contamination_atlas/metadata/atlas_wells_result_summary.html

In [77]:
wells_fc = "atlas_contamination_wells_result_sum"
showFieldinfo(wells_fc)

*Summary*
 Fields in the Point Feature Class: atlas_contamination_wells_result_sum

Name                 Type         Length   Unique_Values
 OBJECTID             | OID        | 4     | 17443     
 Shape                | Geometry   | 0     | 12914     
 facility_c           | String     | 20    | 303       
 facility_i           | Double     | 8     | 303       
 facility_t           | String     | 20    | 7         
 facility_n           | String     | 60    | 303       
 sys_loc_co           | String     | 20    | 11246     
 loc_name             | String     | 80    | 3795      
 loc_type             | String     | 20    | 2         
 loc_type_2           | String     | 50    | 18        
 well_statu           | String     | 20    | 6         
 geologic_u           | String     | 254   | 73        
 depth_of_w           | Double     | 8     | 1145      
 bottom_of_           | Double     | 8     | 1202      
 surf_elev            | String     | 20    | 6199      
 latest_sam        

In [46]:
# This also seems to be a case where we do not have unique values for the ID field that stores locations i.e, sys_loc_co
# Means that the water monitoring location IDs are being duplicated in the table

wells_fc = "atlas_contamination_wells_result_sum"
from collections import Counter

ids = []
with arcpy.da.SearchCursor(wells_fc, ["sys_loc_co"]) as cursor:
    for row in cursor:
        ids.append(row[0])

            # For each item grab only those item in the Counter dictionary (item,count) if the count >1 
Duplicates = [item for item, count in Counter(ids).items() if count >1]

print(f"Duplicate id count: {len(Duplicates)}")
        


Duplicate id count: 4968


In [47]:
## Why do we use Counter(ids).items() and not just loop over Counter(id)?
# To get the key:value pair, having both enable use to filter keys by their counts

from collections import Counter

ids= ["A11", "B22", "B22", "D12","E77","E77","E77"]

for key in Counter(ids): # looping over Counter(ids) directly
    print(key) # just holds keys

dups= Counter(ids).items() # Creates a counter object with counts of the key
print("\n",dups)

dups= [key for key,count in Counter(ids).items() if count >1] # now we can use count to filter
print("\n", dups)

A11
B22
D12
E77

 dict_items([('A11', 1), ('B22', 2), ('D12', 1), ('E77', 3)])

 ['B22', 'E77']


In [ ]:
### What do duplicate well location IDs actually mean? 
# It means that each well may have different timepoints or some other attribute that is different across the different entries with same well ID
### why do we care?
    # Goal:  to analyze those wells with contaminant exceeding standards for each unique City/township for the most recent sampling date 
    # although multiple wells at the same exact position within a CTU may not visually change the conclusion
    # The total records counts and analyses are easier to understand and more meaningful if each row in our dataset represents a unique well grouped by CTU   

In [ ]:
# we need to understand these duplicates better, are they simply different sampling dates?

wells_fc = "atlas_contamination_wells_result_sum"


id_date_counter = Counter() # counter can be used to set up an empty dictionary object that counts keys

with arcpy.da.SearchCursor(wells_fc,["sys_loc_co","latest_sam"]) as cursor:
    for row in cursor:
        key= (row[0], row[1]) # key created as a tuple of two fields: well id and date
        id_date_counter[key] += 1  # when we see the same combo well+date, add 1



print("\nDuplicate Well ID Sampling Date Combinations\n", len(id_date_counter))

for key,count in id_date_counter.items():
    if count>1:
        print(f"sys_loc_code {key[0]} on date {key[1]} occurs {count} times")



Duplicate Well ID Sampling Date Combinations
 11246
sys_loc_code 257221 on date 2007-03-22 00:00:00 occurs 2 times
sys_loc_code MNPCAP17884 on date 2008-03-18 00:00:00 occurs 2 times
sys_loc_code 122194 on date 2024-06-05 00:00:00 occurs 2 times
sys_loc_code 1000029195 on date 2022-08-22 00:00:00 occurs 2 times
sys_loc_code 542025 on date 2019-05-16 00:00:00 occurs 2 times
sys_loc_code 175188 on date 2006-05-25 00:00:00 occurs 3 times
sys_loc_code 457693 on date 2022-04-21 00:00:00 occurs 2 times
sys_loc_code 257614 on date 2022-01-25 00:00:00 occurs 2 times
sys_loc_code 710101 on date 2024-08-15 00:00:00 occurs 2 times
sys_loc_code 257704 on date 2019-05-28 00:00:00 occurs 2 times
sys_loc_code 593137 on date 2022-06-02 00:00:00 occurs 3 times
sys_loc_code 645631 on date 2021-10-25 00:00:00 occurs 2 times
sys_loc_code 761657 on date 2019-09-24 00:00:00 occurs 2 times
sys_loc_code 340455 on date 2019-10-31 00:00:00 occurs 2 times
sys_loc_code 257279 on date 2006-04-27 00:00:00 occurs 2

In [ ]:
## No they are not simply different sampling dates: sampling date data is redundant (i.e., repeating same sampling dates for the same well)
# we only need the most recent date (one of the duplicates well dates) for each well location

In [ ]:
## Are there unique dates that repeat for the same well ID?

from collections import defaultdict

well_dates = defaultdict(set) # initializes a dictionary that automatically creates a new empty set() for each new key added

with arcpy.da.SearchCursor(wells_fc, ["sys_loc_co", "latest_sam"]) as cursor:
    for row in cursor:
        
        # if the row[0](sys_loc_co) already exists add the date to the set,
        well_dates[row[0]].add(row[1]) # if sys_loc_co does not exist in the dictionary, create a new sys_loc_co ID entry and add its dates

# count wells with different dates
print("\n Well with multiple different sample dates:\n")
for code,dates in well_dates.items():
    
    # count set of date values stored as the dictionary value 
    if len(dates)>1: # if the count of dates for that key is >1 print the well code and its dates--> these are wells with multiple sampling dates
        print(f"Well {code} {sorted(dates)}")



 Well with multiple different sample dates:



In [ ]:
## Lack of result here consistent with idea that all duplicate well entries have the same dates
# now we can safely pick one well ID and one date without worrying that duplicates prevent us from selecting the most recent one

In [ ]:

## To ensure we correctly group wells by unique CTU locations
# Define a unique CTU:
#   A spatially unique CTU is a unique combination of CTU Feature ID (GNIS_FEATURE_ID) + County ID (COUNY_GNIS_ID) 

## We need to 
# 1) Create a new field that will mark each spatially unique CTU in the CTU layer
# 2) Join the modified CTU layer with the Wells Data

# Now we have a field in the wells data that can be used to group wells by CTU polygon

### **Add a New Field**
#### `arcpy.management.AddField()`
- use to add new fields to a feature class attributes
- arcpy.management.AddField(input_data, field_name, field_type, {field_alias},{field_length} , only applies to text--Default is 255, {template})
    - field types include: SHORT, DOUBLE, TEXT , DATE, DATEONLY, TIMEONLY, BLOB, RASTER



In [50]:
### Create a new field
fc= "city_township_unorg"
new_field = "Combined_GNIS_IDs"

try:
    # use a list comprehension to check for the new field
    if new_field not in [f.name for f in arcpy.ListFields(fc)]:

        ## Add a new field
        arcpy.management.AddField(fc, new_field,"TEXT",field_length=50)
        print(arcpy.GetMessages(0))

except arcpy.ExecuteError:
    print("Arcpy Error occurred", arcpy.GetMessages(2))
except Exception as e:
    print("Error Occurred", {e}, type(e).__name__)


Start Time: Wednesday, March 26, 2025 11:05:49 PM
Adding Combined_GNIS_IDs to city_township_unorg...
Succeeded at Wednesday, March 26, 2025 11:05:49 PM (Elapsed Time: 0.03 seconds)


### **Edit Feature Class and Table attributes with UpdateCursor**
#### `arcpy.da.UpdateCursor()`
- use to add data to a feature class or table
- UpdateCursor (in_table, field_names, {where_clause}, {spatial_reference}, {explode_to_points}, {sql_clause}, {datum_transformation}, {explicit}, {spatial_filter}, {spatial_relationship}, {search_order})
    - Works similar to SearchCursor in that you are accessing the rows of data for a table
    - Key difference is how to apply changes 
        - cursor.updateRow(row)

- Example:
    - with arcpy.da.SearchCursor() we read data and store it
        - for row in cursor:
            - some_data = row[0]
    - with arcpy.da.UpdateCursor() reads data and also write data to the row
        - for row in cursor:
            - some_data = row[0]
            - row[1] = some_data
            - cursor.updateRow(row)

In [ ]:
### Update the table with the new field that combines two ID fields to create a combined unique ID field
fc= "city_township_unorg"
new_field = "Combined_GNIS_IDs"
field_names= ['GNIS_FEATU', 'COUNTY_GNI', new_field] 

try:
    with arcpy.da.UpdateCursor(fc, field_names) as cursor:

        for row in cursor:
            gnis_id = str(row[0]) if row[0] is not None else "NULL" # if value is not None store is as a string -- allows us combine fields
            county_id = str(row[1]) if row[1] is not None else "NULL"
            row[2]= f"{gnis_id}_{county_id}" # sets the new field value

            cursor.updateRow(row) # applies the changes to the row 

    print("Successfully Updated Records with new ID")
    print(arcpy.GetMessages(0))
    
except arcpy.ExecuteError:
    print("Arcy Error: ", arcpy.GetMessages(2))
except Exception as e:
    print("Error occured", e, type(e).__name__)


Successfully Updated Records with new ID
Start Time: Wednesday, March 26, 2025 11:05:49 PM
Adding Combined_GNIS_IDs to city_township_unorg...
Succeeded at Wednesday, March 26, 2025 11:05:49 PM (Elapsed Time: 0.03 seconds)


In [ ]:
### Check the uniqueness of the new combined ID field 

fc= "city_township_unorg"
field_name= "Combined_GNIS_IDs"

unique_ids= set()
try:
    # Iterate through the table and collect the unique values in the new field 
    with arcpy.da.SearchCursor(fc, field_name) as cursor:

        for row in cursor:
            if row[0] is not None: # only non null ids to the set
                unique_ids.add(row[0])
    print(arcpy.GetMessages(0))

except arcpy.ExecuteError:
    print("ArcPy Error", arcpy.GetMessages(2))
except Exception as e:
    print("Error occurred", e , type(e).__name__)

print("Unique records", len(unique_ids))


Start Time: Tuesday, April 1, 2025 3:36:40 PM
Row Count = 2744
Succeeded at Tuesday, April 1, 2025 3:36:40 PM (Elapsed Time: 0.00 seconds)
Unique records 2743


In [85]:
##  Validate if the new field is unique for each record

# Get the record counts for the feature class
record_counts= int(arcpy.management.GetCount(fc)[0])

if len(unique_ids) == record_counts:
    print("Each record has unique a value for the new ID field", field_name)
else:
    print("There are duplicate IDs in the field", field_name)


There are duplicate IDs in the field Combined_GNIS_IDs


In [ ]:
### This occurs because we inserted a WKT geometry with all blank values for all fields
# This was a test feature. Aside from that the 1 duplicate with a Null Id , all others are unique

In [ ]:
## Arcpy documentation (kept for reference)

from IPython.display import IFrame # a module for controlling notebook outputs, allows you to embed images, video, webpages with the Iframe function

# ArcGIS Pro documentation URL for a specific tool
tool_url = "https://pro.arcgis.com/en/pro-app/latest/tool-reference/analysis/spatial-join.htm"

# Display the documentation inside Jupyter Notebook
IFrame(tool_url, width="100%", height="600px") # iframe can be used to display local or online webpages, documents, reports, visualizations , videos

### **Spatial Join Analysis**

#### `arcpy.analysis.SpatialJoin()`
- **arcpy.analysis.SpatialJoin(target_features, join_features, out_feature_class, {join_operation}, {join_type}, {field_mapping}, {match_option}, {search_radius}, {distance_field_name}, {match_fields})**
    - Target features : features to be enriched with attributes
    - Join features: features contributing attribute data
- Key optional arguments:
    - join operation: Type of join either JOIN_ONE_TO_ONE or JOIN_ONE_TO_MANY. default one-to-one
    - join type : Number of target records to keep either 'KEEP_ALL'(default, keep all records) or 'KEEP_COMMON' (only matched records are kept). Default keeps all fields
    - field mapping: Use this to customize output layer fields, add, delete, rename, re-order,change properties, and combine fields
    - match option: Intersect, Within, Contains, Closest, Closest_Geodesic,Are_identical_to, Largest_overlap...and more!
    - search_radius: distance to search for matches
    - match fields: pairs of fields to restrict the join to only those fields with matching values

In [54]:
## Goal:  Group Water contamination point data by city/township
# Will do a spatial join to add city/township polygon layer attribute data to the groundwater water contamination points

try:
                                    ## target                           # join features
    arcpy.analysis.SpatialJoin("atlas_contamination_wells_result_sum", "city_township_unorg",
                            "MN_WaterContamination_SpatialJoin","JOIN_ONE_TO_ONE","KEEP_ALL","","INTERSECT")            
    print("Successfully joined the features", arcpy.GetMessages(0))

except arcpy.ExecuteError:
    print("Error occured during join: ", arcpy.GetMessages(2))


Successfully joined the features Start Time: Wednesday, March 26, 2025 11:09:16 PM
Succeeded at Wednesday, March 26, 2025 11:09:17 PM (Elapsed Time: 1.35 seconds)


In [ ]:
# - New fields created in the joined output layer
#     - Join_Count - counts the number of times a target feature matched to a join feature
#         - For a one-to-one join, join_count will be a 1 or 0, where a 0 means no matches found
#         - if their are overlapping join features (e.g,CTU), AND we choose a one-to-many join_operation, Join_Count can be >1 (e.g., 1 well matches 2 CTU)
#     - TARGET_FID - Essentially is the Object ID with a new name. Tells you which original feature each record corresponds to
#     - JOIN_FID (if using, JOIN_ONE_TO_MANY) : The ID of the join feature that matched to the target feature.
#         - Will have separate rows for each match between target and join features. Evaluate Target_FID and JOIN_ID to see the matches
#         - JOIN_FID= -1 , no matches to the target feature

### Result

<img src= "{static}/images/SpatialJoinResult_JoinCount0.png" alt ="ArcGIS showing Map of Spatial Join Result for well sites enriched with county/township location attributes" style= " width 400px; height: 400px;">

- This is the result of joining the Water Contamination Monitoring Point Feature Class and County Township Unorganized (CTU) Territory Polygon Feature class
- This new Feature class shows Join_Count = 1 for all features except one , meaning all but one water monitoring site was matched to a CTU location  
    - Makes sense for this one to not have matched since there is no CTU polygon in Iowa (where the well sample site is located)
    - All fields for the unmatched out-of-state well show Null values for all the added CTU attribute fields, as expected
    - If needed, we can remove the non-MN state well sampling site from the dataset